Generating Study area figure

In [ ]:
# -------- USER INPUTS --------
nc_file = r"CHL_Complete_to_Jan2026.nc"
variable_name = "CHL"
output_file = r"Figure_1.pdf"   # changed to .pdf

lon_min, lon_max = 89.0, 92.5
lat_min, lat_max = 20.5, 22.8
time_index = 0

bathy_file = r"gebco_2026.nc"

# -------- IMPORTS --------
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# [NEW] Embed fonts properly in the PDF instead of leaving them as
# non-embedded Type 3 fonts — satisfies the journal's vector-format
# requirement ("embedding the font or saving text as graphics").
plt.rcParams['pdf.fonttype'] = 42   # TrueType embedding
plt.rcParams['ps.fonttype'] = 42

def _dd_to_dms(dd, is_lon):
    hemi = ('E' if dd >= 0 else 'W') if is_lon else ('N' if dd >= 0 else 'S')
    dd = abs(dd)
    d = int(dd)
    m_float = (dd - d) * 60
    m = int(m_float)
    s = round((m_float - m) * 60)
    if s == 60:
        s = 0
        m += 1
    if m == 60:
        m = 0
        d += 1
    return f"{d}\u00b0{m:02d}\u2032{s:02d}\u2033{hemi}"

class FullDMSLonFormatter(mticker.Formatter):
    def __call__(self, value, pos=None):
        return _dd_to_dms(value, is_lon=True)

class FullDMSLatFormatter(mticker.Formatter):
    def __call__(self, value, pos=None):
        return _dd_to_dms(value, is_lon=False)

try:
    import cmocean
    cmap = cmocean.cm.algae
except:
    cmap = "viridis"

# -------- LOAD DATA --------
ds = xr.open_dataset(nc_file)
data = ds[variable_name]
lon_name = [c for c in ds.coords if 'lon' in c.lower()][0]
lat_name = [c for c in ds.coords if 'lat' in c.lower()][0]
subset = data.sel({lon_name: slice(lon_min, lon_max), lat_name: slice(lat_min, lat_max)})
if "time" in data.dims:
    data_plot = data.isel(time=time_index)
    subset_plot = subset.isel(time=time_index)
else:
    data_plot = data
    subset_plot = subset

# -------- FIGURE --------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 12,
    "font.weight": "bold",
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

COLOR_LAND = "#ffffff"
COLOR_LAND_EDGE = "#4d4d4d"
COLOR_BORDER = "#3a3a3a"
COLOR_GRID = "#ffffff"
COLOR_OCEAN = "#a9d3e8"

# [CHANGED] Single-panel figure — no locator inset, so a slightly
# smaller/more map-proportioned canvas suits a standalone panel better.
fig = plt.figure(figsize=(9, 7))

# ===============================
# SINGLE MAP — NE Bay of Bengal study area
# [CHANGED] ax2 now fills the whole figure, no inset panel alongside it
# ===============================
ax2 = fig.add_axes([0.1, 0.1, 0.85, 0.85], projection=ccrs.PlateCarree(), zorder=1)
ax2.set_extent([lon_min, lon_max, lat_min, lat_max])

ax2.add_feature(cfeature.OCEAN.with_scale('10m'), facecolor=COLOR_OCEAN, zorder=0)
ax2.add_feature(cfeature.LAND.with_scale('10m'), facecolor=COLOR_LAND,
                 edgecolor=COLOR_LAND_EDGE, linewidth=0.5, zorder=1)
ax2.add_feature(cfeature.COASTLINE.with_scale('10m'), linewidth=0.7, zorder=2)
ax2.add_feature(cfeature.BORDERS.with_scale('10m'), linewidth=0.9, edgecolor=COLOR_BORDER, zorder=2)

ax2.scatter(91.40, 22.28, color='blue', s=14, transform=ccrs.PlateCarree(), zorder=5)
ax2.text(90.70, 21.18, "Coastal waters of Bangladesh.\n Northeastern Bay of Bengal", fontsize=12, fontweight='bold',
         family='Times New Roman', ha='center', transform=ccrs.PlateCarree())
ax2.text(91.50, 22.20, "GBM estuary", fontsize=12, fontweight='bold',
         family='Times New Roman', ha='center', transform=ccrs.PlateCarree())
ax2.text(92.10, 22.12, "Chittagong", fontsize=12, fontweight='bold',
         family='Times New Roman', ha='center', transform=ccrs.PlateCarree())

if bathy_file:
    bathy = xr.open_dataset(bathy_file)
    elev = bathy['elevation']
    elev_sub = elev.sel(lon=slice(lon_min, lon_max), lat=slice(lat_min, lat_max))
    depth_levels = [-1000, -500, -200, -100, -50, -10]
    cs = ax2.contour(
        elev_sub['lon'], elev_sub['lat'], elev_sub,
        levels=sorted(depth_levels),
        colors='black', linewidths=0.5,
        transform=ccrs.PlateCarree(), zorder=3
    )
    ax2.clabel(cs, inline=True, fontsize=7, fmt=lambda x: f"{abs(int(x))}m")

gl2 = ax2.gridlines(draw_labels=True, linewidth=0.7, linestyle='--', color=COLOR_GRID)
gl2.top_labels = True
gl2.right_labels = True
gl2.xformatter = FullDMSLonFormatter()
gl2.yformatter = FullDMSLatFormatter()
gl2.xlabel_style = {"size": 9, "weight": "bold", "family": "Times New Roman"}
gl2.ylabel_style = {"size": 9, "weight": "bold", "family": "Times New Roman"}
gl2.xlocator = mticker.MaxNLocator(4)
gl2.ylocator = mticker.MaxNLocator(4)

# ===============================
# SAVE — vector PDF, no dpi ceiling since it's true vector output
# ===============================
plt.savefig(output_file, format='pdf', bbox_inches='tight')
plt.show()
plt.close()

print("Final vector PDF study-area figure saved with updated coordinate limits.")